# Nigeria: portability validation

This notebook checks whether the monitoring structure can retain a second country’s source, currency, geography level, product, unit, and collection month without mixing it into the Colombia evidence.

> **Price-only boundary:** these Nigeria foods do not yet have reviewed nutrition profiles in this pilot. The analysis shows price variation only and must not generate a food-substitution recommendation.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd() / 'colombia_sipsa_pilot', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'data' / 'processed' / 'nigeria_nbs_zone_price_ranges_march_may_2026.csv').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from the repository or colombia_sipsa_pilot folder.')

ROOT = find_project_root()
PROCESSED = ROOT / 'data' / 'processed'
ROOT

PosixPath('/Users/ginamancuso/.codex/.chatgpt-projects/g-p-6a847acb9d1c8191be127194068698c8/publish_repo/colombia_sipsa_pilot')

## 1. Preserve the source context

These records are Nigeria National Bureau of Statistics zone-level monthly averages. They are not Colombia wholesale market observations, so the two datasets remain separate.

In [2]:
prices = pd.read_csv(PROCESSED / 'nigeria_nbs_zone_prices_march_may_2026.csv', parse_dates=['date'])
assert set(prices['country']) == {'Nigeria'}
assert set(prices['geography_level']) == {'Zone'}
assert set(prices['currency']) == {'NGN'}
assert set(prices['unit']) == {'1 kg'}

display(prices.head(12))
print(f"Loaded {len(prices)} Nigeria zone-price observations for {prices['item'].nunique()} foods across {prices['date'].nunique()} months.")

,date,country,geography_level,zone,item,unit,currency,price_ngn,source_url
0,2026-03-01,Nigeria,Zone,NORTH CENTRAL,Beans Brown,1 kg,NGN,1229.762524,https://microdata.nigerianstat.gov.ng/index.ph...
1,2026-03-01,Nigeria,Zone,NORTH EAST,Beans Brown,1 kg,NGN,868.792427,https://microdata.nigerianstat.gov.ng/index.ph...
2,2026-03-01,Nigeria,Zone,NORTH WEST,Beans Brown,1 kg,NGN,851.114018,https://microdata.nigerianstat.gov.ng/index.ph...
3,2026-03-01,Nigeria,Zone,SOUTH EAST,Beans Brown,1 kg,NGN,1615.867271,https://microdata.nigerianstat.gov.ng/index.ph...
4,2026-03-01,Nigeria,Zone,SOUTH SOUTH,Beans Brown,1 kg,NGN,1762.489902,https://microdata.nigerianstat.gov.ng/index.ph...
5,2026-03-01,Nigeria,Zone,SOUTH WEST,Beans Brown,1 kg,NGN,1770.571223,https://microdata.nigerianstat.gov.ng/index.ph...
6,2026-03-01,Nigeria,Zone,NORTH CENTRAL,Garri white,1 kg,NGN,670.164718,https://microdata.nigerianstat.gov.ng/index.ph...
7,2026-03-01,Nigeria,Zone,NORTH EAST,Garri white,1 kg,NGN,733.330428,https://microdata.nigerianstat.gov.ng/index.ph...
8,2026-03-01,Nigeria,Zone,NORTH WEST,Garri white,1 kg,NGN,763.091428,https://microdata.nigerianstat.gov.ng/index.ph...
9,2026-03-01,Nigeria,Zone,SOUTH EAST,Garri white,1 kg,NGN,942.035492,https://microdata.nigerianstat.gov.ng/index.ph...


Loaded 36 Nigeria zone-price observations for 2 foods across 3 months.


## 2. Review monthly zone-price ranges

For each food and month, the low-to-high zone range is shown with the corresponding zones. This is an auditable price signal, not a causal explanation for differences.

In [3]:
ranges = pd.read_csv(PROCESSED / 'nigeria_nbs_zone_price_ranges_march_may_2026.csv', parse_dates=['date'])
ranges['gap_pct_of_low'] = ranges['gap_pct_of_low'].round(1)
display(ranges.sort_values(['item', 'date']))

beans = ranges.loc[ranges['item'].eq('Beans Brown')].sort_values('date')
assert beans['gap_ngn'].between(900, 930).all(), 'Expected the verified Beans Brown gap pattern.'
print('Beans Brown had a high-to-low zone difference of roughly NGN 914–919 per kg in each sampled month.')

,date,item,unit,lowest_price_ngn,highest_price_ngn,gap_ngn,gap_pct_of_low,lowest_zone,highest_zone
0,2026-03-01,Beans Brown,1 kg,851.114018,1770.571223,919.457205,108.0,NORTH WEST,SOUTH WEST
2,2026-04-01,Beans Brown,1 kg,871.792427,1787.089452,915.297025,105.0,NORTH EAST,SOUTH WEST
4,2026-05-01,Beans Brown,1 kg,876.459093,1790.709020,914.249926,104.3,NORTH EAST,SOUTH WEST
1,2026-03-01,Garri white,1 kg,670.164718,942.681334,272.516616,40.7,NORTH CENTRAL,SOUTH SOUTH
3,2026-04-01,Garri white,1 kg,673.878543,944.578108,270.699564,40.2,NORTH CENTRAL,SOUTH EAST
5,2026-05-01,Garri white,1 kg,676.213167,947.078121,270.864954,40.1,NORTH CENTRAL,SOUTH EAST


Beans Brown had a high-to-low zone difference of roughly NGN 914–919 per kg in each sampled month.


## Takeaway

The portability check demonstrates a reusable review-first schema across country, currency, source format, geography, and unit. Before this data can support nutrition-aware food alternatives, locally appropriate nutrition profiles and human review must be added.